In [9]:
# Updated for local environment - no need for Google Drive mounting
import os
import sys
import subprocess

# Get the current working directory using pwd
workspace_path = subprocess.check_output(['pwd'], universal_newlines=True).strip()
os.chdir(workspace_path)

# Add the workspace to Python path for imports
sys.path.append(workspace_path)

print(f"Working directory: {workspace_path}")

Working directory: /Users/omar-sharif/Library/CloudStorage/GoogleDrive-omar.sharif.gr@dartmouth.edu/Shared drives/REGen


In [10]:
# Install required packages (comment out if already installed)
# !pip install langchain-openai
# !pip install langchain
# !pip install -U langchain-huggingface
# !pip install transformers
# !pip install bert-score
# !pip install -U sentence-transformers
# !pip install spacy

# Download spacy model if not already downloaded
# !python -m spacy download en_core_web_sm

import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Download NLTK resources (comment out if already downloaded)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

print("NLTK resources loaded successfully!")

NLTK resources loaded successfully!


[nltk_data] Downloading package wordnet to /Users/omar-
[nltk_data]     sharif/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [11]:
import os, json
import numpy as np
from argparse import ArgumentParser
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
from pprint import pprint
from datetime import datetime
import copy
import pickle
from ast import literal_eval
import re, string

# Add src to path to import from regen package
current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else workspace_path
src_path = os.path.join(current_dir, 'src')
sys.path.insert(0, src_path)

# Import functions from existing modules
from regen.evaluate import (
    read_json_file,
    get_processed_predictions_and_schema,
    get_predictions_for_complex_matching,
    get_predictions_for_results,
    doing_exact_match,
    clac_sbert_score,
    calculating_semantic_score,
    relaxed_match_thresholding,
    getting_exact_relaxed_match_predictions_dictionary,
    pair_matching_prompt_chain,
    complex_pair_matching,
    getting_complex_match_pairs,
    doing_complex_matching,
    getting_complex_match_predictions_dictionary,
    getting_role_wise_scores
)

print("Functions loaded successfully!")



Functions loaded successfully!


In [13]:
import spacy

# Load spaCy English model
try:
    nlp = spacy.load("en_core_web_sm")
    print("spaCy model loaded successfully!")
except OSError:
    print("spaCy model not found. Please install it with: python -m spacy download en_core_web_sm")
    nlp = None

spaCy model loaded successfully!


In [14]:


def get_head_nouns_string(text):
    doc = nlp(text)
    head_nouns = []
    for chunk in doc.noun_chunks:
        head_nouns.append(chunk.root.text.lower())  # get head of each noun phrase
    # Remove duplicates, preserve order, and return as a space-separated string
    head_nouns_unique = list(dict.fromkeys(head_nouns))
    return ' '.join(head_nouns_unique)

def doing_head_noun_phrase_match(predictions, unique_roles):
    new_pred_dictionary = []

    for role in unique_roles:
        print(f"-----{role}------")
        cnt = 0
        for dt in predictions:
            if(dt['role']!=role): continue #skipping the roles that does not match
            new_dt = copy.deepcopy(dt)
            normalized_actual_labels = dt['initial-ground-truth']  #this labels and predictions are normalized
            normalized_predictions = dt['initial-predictions']

            #needed for sanity printing. can be commented later
            initial_labels = copy.deepcopy(normalized_actual_labels)
            initial_predictions = copy.deepcopy(normalized_predictions)

            hm_pair = []
            for p in normalized_predictions[:]:
                if(len(normalized_predictions)<=0 or len(normalized_actual_labels)<=0):
                    break #if any of the list becomes zero then no need for comparison
                for g in normalized_actual_labels[:]:
                    if(len(normalized_predictions)<=0 or len(normalized_actual_labels)<=0):
                        break #if any of the list becomes zero then no need for comparison
                    p_head_noun = get_head_nouns_string(p)
                    g_head_noun = get_head_nouns_string(g)
                    if (p_head_noun == g_head_noun) and len(p_head_noun)>0 and len(g_head_noun)>0 and (g in normalized_actual_labels) and (p in normalized_predictions): #if we find a match
                      print(role, "||", p,"||", g)
                      normalized_actual_labels.remove(g) ##removing this item from the ground-truth list as it predicted correctly
                      normalized_predictions.remove(p) ##removing from prediction list as it predicted correctly.
                      hm_pair.append((p, g))

                      #print(role, "||", p_head_noun,"||", g_head_noun)

            #this is for sanity printing// can be commented later
            # if(len(hm_pair)>0):
            #     print(len(hm_pair), hm_pair)
            #     cnt += len(hm_pair)
            #     print("GT:", len(normalized_actual_labels),"---", normalized_actual_labels, "---",  initial_labels)
            #     print("PD:",len(normalized_predictions),"---", normalized_predictions,"---", initial_predictions)

            #new list of ground-truth and predictions after exact match
            new_dt['after-HM-ground-truth'] = normalized_actual_labels
            new_dt['after-HM-predictions'] = normalized_predictions
            new_dt['HM-pairs'] = hm_pair

            ##creating new dictionary with exact-match
            new_pred_dictionary.append(new_dt)

        #print(role, cnt)
    return new_pred_dictionary

def getting_head_noun_match_phrase_predictions_dictionary(path, model_name, prompt_type, dataset_name, version):
    initial_predictions, event_schema = get_processed_predictions_and_schema(path, model_name,
                                                                                  prompt_type, dataset_name, version)
    unique_roles = list({d['role'] for d in initial_predictions if 'role' in d})
    after_hm_predictions_dictionary = copy.deepcopy(doing_head_noun_phrase_match(copy.deepcopy(initial_predictions), unique_roles))
    #print(after_em_predictions_dictionary[0].keys())
    return after_hm_predictions_dictionary

print("Head noun phrase matching functions loaded successfully!")


Head noun phrase matching functions loaded successfully!


In [15]:
def get_HM_pred(path, dataset_name, model_name, prompt_type, version):
        print(model_name, dataset_name)
        predictions = getting_head_noun_match_phrase_predictions_dictionary(path, model_name,
                                                                  prompt_type, dataset_name, version)
        #saving the final-predictions in the result folder
        result_path = os.path.join(path, 'Result')
        # Create directory if it doesn't exist
        os.makedirs(os.path.join(result_path, dataset_name), exist_ok=True)
        file_name = f'{model_name}-{prompt_type}-after-HM-predictions-{dataset_name}-v{version}.json'
        with open(os.path.join(result_path, dataset_name, file_name), 'w') as json_file:
              json.dump(predictions, json_file, indent=4)
        return predictions

In [16]:
path = workspace_path
dataset_names = ["DiscourseEE", 'PHEE', 'RAMS', 'GENEVA', 'DocEE', 'WikiEvents']

versions = [1]
prompt_types = ['zero-shot', 'cot']
model_names = ['Phi-3.5', 'Gemma1.1-7B', 'Mixtral-8x7B',  'Llama3.1-70B', 'GPT-4o']

for dataset_name in dataset_names:
    for model_name in model_names:
        for prompt_type in prompt_types:
            for version in versions:
                predictions = get_HM_pred(path, dataset_name, model_name, prompt_type, version)

Phi-3.5 DiscourseEE
-----trigger------
-----medications------
medications || suboxone || suboxone
medications || subutex || subutex
medications || suboxone || suboxone
-----relapse-intervention------
relapse-intervention || support group help || quitting with support group help
-----existing/current-medications------
existing/current-medications || suboxone || suboxone
existing/current-medications || suboxone || suboxone
existing/current-medications || subs || subs
existing/current-medications || suboxone || suboxone
existing/current-medications || subutex || subutex
existing/current-medications || suboxone || suboxone
-----current-dosage------
-----relapse-event------
relapse-event || feeling very sick of kratom || taking kratom
relapse-event || inducing from heroin is supposedly much easier that switching from fentanyl || taking heroin
-----manner------
-----treatment------
treatment || suboxone || 8mg suboxone
treatment || suboxone || taking suboxone
-----tapering-event------
-----t

## Reading the predictions and get results

In [21]:
def overall_score_on_whole_data(predictions):
    '''
    Function for calculating head nour phrase match socres.
    '''
    results = {}
    gt_cnt, pd_cnt, na_hm, np_hm = 0, 0, 0, 0
    hm_cnt = 0

    for dt in predictions:
        #sanity_print(dt)
        gt_cnt += len(dt['initial-ground-truth'])
        pd_cnt += len(dt['initial-predictions'])

        #number of arguments predicted correct from the ground-truth and predicted list of arguments under exact-match
        na_hm += len(dt['initial-ground-truth']) - len(dt['after-HM-ground-truth'])
        np_hm += len(dt['initial-predictions']) - len(dt['after-HM-predictions'])

        hm_cnt += len(dt['HM-pairs'])
    epsilon = 1e-10
    total_cnt = hm_cnt
    # print(role)
    #print(f"GT: {gt_cnt}, PD: {pd_cnt}, HM:{na_hm, np_hm}, HM: {total_cnt}")

    #exact-match precision, recall, f1-score calculation
    hm_precision = np_hm/max(pd_cnt, epsilon)
    hm_recall = na_hm/max(gt_cnt, epsilon)
    hm_f1 = (2 * hm_precision * hm_recall)/max((hm_precision + hm_recall), epsilon)


    results = {
        'HM-precision': round(hm_precision*100, 2),
        'HM-recall': round(hm_recall*100, 2),
        'HM-f1': round(hm_f1*100, 2),

        'ground-truth-count': gt_cnt,
        'prediction-count': pd_cnt,
        'HM-count': hm_cnt,
    }
    #pprint(results)
    return results

results = overall_score_on_whole_data(predictions)
pprint(results)

{'HM-count': 117,
 'HM-f1': 21.55,
 'HM-precision': 19.09,
 'HM-recall': 24.74,
 'ground-truth-count': 473,
 'prediction-count': 613}


In [22]:
path = workspace_path
dataset_names = ["DiscourseEE", 'PHEE', 'RAMS', 'GENEVA', 'DocEE', 'WikiEvents']

versions = [1]
prompt_types = ['zero-shot', 'cot']
model_names = ['Phi-3.5', 'Gemma1.1-7B', 'Mixtral-8x7B',  'Llama3.1-70B', 'GPT-4o']

all_results = {}
for dataset_name in dataset_names:
    p, r, f1 = [], [], []
    for version in versions:
        for prompt_type in prompt_types:
            for model_name in model_names:
                print(model_name, dataset_name, prompt_type)
                result_path = os.path.join(path, 'Result')
                file_name = f'{model_name}-{prompt_type}-after-HM-predictions-{dataset_name}-v{version}.json'
                predictions = json.load(open(os.path.join(result_path, dataset_name, file_name)))
                results = overall_score_on_whole_data(predictions)
                key = f'{dataset_name}-{model_name}-{prompt_type}-v{version}'
                all_results[key] = results
                p.append(results['HM-precision'])
                r.append(results['HM-recall'])
                f1.append(results['HM-f1'])
    print(dataset_name, np.mean(p), np.mean(r), np.mean(f1))

Phi-3.5 DiscourseEE zero-shot
Gemma1.1-7B DiscourseEE zero-shot
Mixtral-8x7B DiscourseEE zero-shot
Llama3.1-70B DiscourseEE zero-shot
GPT-4o DiscourseEE zero-shot
Phi-3.5 DiscourseEE cot
Gemma1.1-7B DiscourseEE cot
Mixtral-8x7B DiscourseEE cot
Llama3.1-70B DiscourseEE cot
GPT-4o DiscourseEE cot
DiscourseEE 11.059000000000001 16.169 13.092000000000002
Phi-3.5 PHEE zero-shot
Gemma1.1-7B PHEE zero-shot
Mixtral-8x7B PHEE zero-shot
Llama3.1-70B PHEE zero-shot
GPT-4o PHEE zero-shot
Phi-3.5 PHEE cot
Gemma1.1-7B PHEE cot
Mixtral-8x7B PHEE cot
Llama3.1-70B PHEE cot
GPT-4o PHEE cot
PHEE 37.217 40.898999999999994 38.938
Phi-3.5 RAMS zero-shot
Gemma1.1-7B RAMS zero-shot
Mixtral-8x7B RAMS zero-shot
Llama3.1-70B RAMS zero-shot
GPT-4o RAMS zero-shot
Phi-3.5 RAMS cot
Gemma1.1-7B RAMS cot
Mixtral-8x7B RAMS cot
Llama3.1-70B RAMS cot
GPT-4o RAMS cot
RAMS 14.373000000000001 19.55 16.455
Phi-3.5 GENEVA zero-shot
Gemma1.1-7B GENEVA zero-shot
Mixtral-8x7B GENEVA zero-shot
Llama3.1-70B GENEVA zero-shot
GPT-4o

In [23]:
all_results.keys()

dict_keys(['DiscourseEE-Phi-3.5-zero-shot-v1', 'DiscourseEE-Gemma1.1-7B-zero-shot-v1', 'DiscourseEE-Mixtral-8x7B-zero-shot-v1', 'DiscourseEE-Llama3.1-70B-zero-shot-v1', 'DiscourseEE-GPT-4o-zero-shot-v1', 'DiscourseEE-Phi-3.5-cot-v1', 'DiscourseEE-Gemma1.1-7B-cot-v1', 'DiscourseEE-Mixtral-8x7B-cot-v1', 'DiscourseEE-Llama3.1-70B-cot-v1', 'DiscourseEE-GPT-4o-cot-v1', 'PHEE-Phi-3.5-zero-shot-v1', 'PHEE-Gemma1.1-7B-zero-shot-v1', 'PHEE-Mixtral-8x7B-zero-shot-v1', 'PHEE-Llama3.1-70B-zero-shot-v1', 'PHEE-GPT-4o-zero-shot-v1', 'PHEE-Phi-3.5-cot-v1', 'PHEE-Gemma1.1-7B-cot-v1', 'PHEE-Mixtral-8x7B-cot-v1', 'PHEE-Llama3.1-70B-cot-v1', 'PHEE-GPT-4o-cot-v1', 'RAMS-Phi-3.5-zero-shot-v1', 'RAMS-Gemma1.1-7B-zero-shot-v1', 'RAMS-Mixtral-8x7B-zero-shot-v1', 'RAMS-Llama3.1-70B-zero-shot-v1', 'RAMS-GPT-4o-zero-shot-v1', 'RAMS-Phi-3.5-cot-v1', 'RAMS-Gemma1.1-7B-cot-v1', 'RAMS-Mixtral-8x7B-cot-v1', 'RAMS-Llama3.1-70B-cot-v1', 'RAMS-GPT-4o-cot-v1', 'GENEVA-Phi-3.5-zero-shot-v1', 'GENEVA-Gemma1.1-7B-zero-shot-

In [24]:
all_results_df = pd.DataFrame.from_dict(all_results, orient='index')
all_results_df

,HM-precision,HM-recall,HM-f1,ground-truth-count,prediction-count,HM-count
DiscourseEE-Phi-3.5-zero-shot-v1,4.46,5.02,4.72,997,1121,50
DiscourseEE-Gemma1.1-7B-zero-shot-v1,12.13,15.95,13.78,997,1311,159
DiscourseEE-Mixtral-8x7B-zero-shot-v1,12.64,20.46,15.63,997,1614,204
DiscourseEE-Llama3.1-70B-zero-shot-v1,13.07,20.06,15.83,997,1530,200
DiscourseEE-GPT-4o-zero-shot-v1,16.05,23.57,19.10,997,1464,235
DiscourseEE-Phi-3.5-cot-v1,8.47,15.05,10.83,997,1772,150
DiscourseEE-Gemma1.1-7B-cot-v1,10.11,16.15,12.44,997,1592,161
DiscourseEE-Mixtral-8x7B-cot-v1,6.07,6.62,6.33,997,1088,66
DiscourseEE-Llama3.1-70B-cot-v1,12.30,17.35,14.40,997,1406,173
DiscourseEE-GPT-4o-cot-v1,15.29,21.46,17.86,997,1400,214
